# Teton vs Bulk RNA-seq plotting notebook

This notebook compares Teton pseudo-bulk (CP10k) to bulk RNA-seq (TPM) and generates:

- Fig1: Well–well scatter per run
- Fig2: Bulk vs Teton scatter per cell line
- Fig3: Delta log-expression heatmaps
***


#### Install `cytoprofiling`: <br>
This notebook uses the `cytoprofiling` package from Element’s GitHub repository. <br>
Run the cell below once to install it into your current Python environment. <br>

In [ ]:
%pip install "git+https://github.com/Elembio/cytoprofiling.git@v1.1.1_python#subdirectory=src/python"

#### Import packages and set plotting defaults: <br>
This cell imports common scientific Python libraries (NumPy/Pandas/Matplotlib) plus `cytoprofiling`.  <br>
It also sets small font sizes and PDF-friendly font embedding so the output figures match the Tech Note style. <br>

In [ ]:

from pathlib import Path
import json, re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import cytoprofiling
from cytoprofiling.normalization import normalize_cells_by_aggregated_counts

# Plotting
plt.rcParams.update({
    "font.size": 6,
    "axes.titlesize": 6,
    "axes.labelsize": 6,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

CM_TO_INCH = 1 / 2.54
FIG_SIZE = (9.2 * CM_TO_INCH, 9.2 * CM_TO_INCH)


#### Configure input files and outputs: <br>

Original files used in this Tech Note are available for download at s3://element-public-data/20260220-Teton-RNAseq/CorrelatingTetonwithRNAseq_datasets.zip. <br>
Update the paths in this section to point to your local copies of: <br>
- Teton `RUNS`: RawCellStats.parquet file for each run <br>
- Teton `PANELS` files: Panel.json file for each run <br>
- Bulk RNA-seq `bulk_path`: salmon.merged.gene_tpm.tsv file <br>
- Outputs folder `output_dir`: where generated tables + PDFs will be written <br>

In [ ]:
RUNS = {
    "HeLa":  Path(r"PATH_TO_HELA_RUN/RawCellStats.parquet"),
    "A549":  Path(r"PATH_TO_A549_RUN/RawCellStats.parquet"),
    "HUVEC": Path(r"PATH_TO_HUVEC_RUN/RawCellStats.parquet"),
}

PANELS = {
    "HeLa":  Path(r"PATH_TO_HELA_PANEL/Panel.json"),
    "A549":  Path(r"PATH_TO_A549_PANEL/Panel.json"),
    "HUVEC": Path(r"PATH_TO_HUVEC_PANEL/Panel.json"),
}

output_dir = Path(r"PATH_TO_OUTPUT_DIR/")
fig_dir = output_dir / "figures"

GROUP_COL = "Well"

bulk_path = Path(r"PATH_TO_BULK_FILE/salmon.merged.gene_tpm.tsv")

BULK_COL_MAP = {
    "HeLa": "Hela_WT",
    "A549": "A549_WT",
    "HUVEC": "HUVEC_WT",
}

#### Helper functions <br>
- bio_mask_from_features: removes non-biological / technical features such as NSB, Unassigned, and Nuclear features
- collapse_feature_batches_to_gene_mean: averages GAPDH counts across multiple batches into a single gene-level value
- load_bulk_tpm: loads the bulk TPM file and ensures numeric values

In [ ]:

def bio_mask_from_features(features):
    idx = pd.Index([str(x) for x in features])
    return ~idx.str.contains(r"(?:NSB|Unassigned|Nuclear)", case=False, regex=True)

def collapse_feature_batches_to_gene_mean(df_well_x_feat, report=False, run_name=None):
    cols = df_well_x_feat.columns.astype(str)
    gene = cols.str.split(".").str[0]

    if report:
        counts = gene.value_counts()
        n_multi = int((counts > 1).sum())
        header = f"[{run_name}] " if run_name else ""
        print(f"{header}Genes appearing in multiple batches: {n_multi}")
    out = df_well_x_feat.T.groupby(gene).mean().T
    return out

def load_bulk_tpm(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")
    df = df.set_index(df.columns[0])
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

#### Process a Teton run into gene-level matrices <br>
- process_run: converts cell-level Teton data into pseudo-bulk matrices (CP10k and log10(CP10k+1))

In [ ]:

def process_run(raw_parquet_path: Path, panel_json_path: Path, group_col="Well", report_batches=False):
    """
    Outputs:
      well_cp10k_gene: wells x genes (CP10k)
      well_log10_gene: wells x genes (log10(CP10k+1))
    """
    with open(panel_json_path, "r") as f:
        j = json.load(f)

    barcode_targets = pd.DataFrame(j["BarcodingTargets"])
    unique_batches = barcode_targets.loc[
        barcode_targets["TargetType"] == "Transcript", "BatchName"
    ].unique()

    df = pd.read_parquet(raw_parquet_path)
    df_filt = cytoprofiling.filter_cells(df).copy()

    constant_columns = {"Cell", "Well", "WellLabel", "Tile", "X", "Y"}
    cols = df_filt.columns.astype(str)

    inc_pat = "|".join(map(re.escape, unique_batches)) if len(unique_batches) else r"$.^"
    include = cols.isin(constant_columns) | cols.str.contains(inc_pat, case=False, regex=True)
    exclude = cols.str.contains(r"(Nuclear|Unassigned)", case=False, regex=True)

    transcript_df = df_filt.loc[:, include & ~exclude].copy()

    # Cell-level batch normalization
    if group_col != "Well":
        _tmp = transcript_df.rename(columns={group_col: "Well"})
    else:
        _tmp = transcript_df

    transcript_df_bn = normalize_cells_by_aggregated_counts(
        _tmp,
        batch_names=list(unique_batches),
        well_names=None,
        aggregation_func=np.nansum,
        normalization_targets=[],
    )

    if group_col != "Well":
        transcript_df_bn = transcript_df_bn.rename(columns={"Well": group_col})

    # Convert to AnnData
    adata = cytoprofiling.cytoprofiling_to_anndata(transcript_df_bn, j)

    # Probe concentration adjustment
    conc = np.asarray(adata.var["probe_concentration"], dtype=float)
    good = np.isfinite(conc) & (conc > 0)
    ref = conc[good].max() if np.any(good) else 1.0
    f = np.ones_like(conc, dtype=float)
    f[good] = ref / conc[good]

    X = np.asarray(adata.X, dtype=float)
    X_adj = X * f

    # CP10k per cell
    totals = X_adj.sum(axis=1)
    X_cp10k = (X_adj / (totals[:, None] + 1e-12)) * 1e4

    # Keep biological genes
    genes = adata.var_names.astype(str)
    m = bio_mask_from_features(genes)
    genes = genes[m]
    X_cp10k = X_cp10k[:, m]

    # Aggregate to well × gene (mean over cells)
    groups = adata.obs[group_col].astype(str).to_numpy()
    uniq = np.unique(groups)

    well_feat = np.zeros((len(uniq), X_cp10k.shape[1]), dtype=float)
    for i, g in enumerate(uniq):
        idx = np.where(groups == g)[0]
        well_feat[i, :] = X_cp10k[idx].mean(axis=0) if idx.size else np.nan

    well_cp10k_feat = pd.DataFrame(well_feat, index=uniq, columns=genes)

    # Collapse batch suffixes -> gene symbol
    well_cp10k_gene = collapse_feature_batches_to_gene_mean(
        well_cp10k_feat,
        report=report_batches,
        run_name=str(raw_parquet_path.parent.name)
    )

    well_log10_gene = np.log10(well_cp10k_gene + 1.0)

    return {
        "well_cp10k_gene": well_cp10k_gene,
        "well_log10_gene": well_log10_gene
    }


# teton_pseudobulk_from_wells: collapses wells into a single pseudo-bulk vector
def teton_pseudobulk_from_wells(results_dict, run_name, wells=None, agg="mean"):
    W = results_dict[run_name]["well_cp10k_gene"]

    if wells is not None:
        missing = [w for w in wells if w not in W.index]
        if missing:
            raise ValueError(f"{run_name}: wells not found: {missing}")
        W = W.loc[wells]

    if agg == "mean":
        v = W.mean(axis=0)
    elif agg == "median":
        v = W.median(axis=0)
    else:
        raise ValueError("agg must be 'mean' or 'median'")

    v.name = run_name
    return v

# align_teton_bulk: intersects genes shared by both datasets and log-transforms the values for plotting
def align_teton_bulk(teton_vec, bulk_tpm_df, bulk_col):
    b = bulk_tpm_df[bulk_col].copy()

    shared = teton_vec.index.astype(str).intersection(b.index.astype(str))
    t2 = teton_vec.loc[shared].astype(float)
    b2 = b.loc[shared].astype(float)

    return np.log10(t2 + 1.0), np.log10(b2 + 1.0)

#### Plotting utilities: <br>
This section contains plotting functions that write PDF figures to the //figures subfolder of the output directory.

In [ ]:
# for Figure 1: scatter plot comparing two wells from the same run
def plot_well_scatter_pdf(df_log10, wa, wb, out_pdf: Path, lims=None):
    x = df_log10.loc[wa].to_numpy(dtype=float)
    y = df_log10.loc[wb].to_numpy(dtype=float)

    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]

    r = np.corrcoef(x, y)[0, 1]
    r2 = r * r

    if lims is None:
        lo = np.nanmin([x.min(), y.min()])
        hi = np.nanmax([x.max(), y.max()])
    else:
        lo, hi = lims

    fig, ax = plt.subplots(figsize=FIG_SIZE)
    ax.scatter(x, y, s=6, alpha=0.7)
    ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=0.8)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)

    ax.set_xlabel(f"{wa} log10(CP10k+1)")
    ax.set_ylabel(f"{wb} log10(CP10k+1)")
    ax.set_title(f"{wa} vs {wb}\nPearson r={r:.4f}, R²={r2:.4f}")

    fig.tight_layout()
    fig.savefig(out_pdf, format="pdf", dpi=300)
    plt.close(fig)

# for Figure 2: bulk vs Teton scatter with Pearson correlation
def plot_teton_vs_bulk_pdf(tlog, blog, cellline, out_pdf: Path, lims=None, pad_frac=0.03):
    x = blog.to_numpy(dtype=float)  # Bulk on X
    y = tlog.to_numpy(dtype=float)  # Teton on Y

    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]

    r = np.corrcoef(x, y)[0, 1]
    r2 = r * r

    if lims is None:
        lo = np.nanmin([x.min(), y.min()])
        hi = np.nanmax([x.max(), y.max()])
    else:
        lo, hi = lims

    span = hi - lo
    pad = span * pad_frac if span > 0 else 0.1
    lo2, hi2 = lo - pad, hi + pad

    fig, ax = plt.subplots(figsize=FIG_SIZE)
    ax.scatter(x, y, s=6, alpha=0.7)
    ax.plot([lo2, hi2], [lo2, hi2], linestyle="--", linewidth=0.8)
    ax.set_xlim(lo2, hi2)
    ax.set_ylim(lo2, hi2)

    ax.set_xlabel("Bulk log10(TPM+1)")
    ax.set_ylabel("Teton log10(CP10k+1) pseudo-bulk")
    ax.set_title(f"{cellline}: Bulk vs Teton\nPearson r={r:.4f}, R²={r2:.4f} (n={x.size})")

    fig.tight_layout()
    fig.savefig(out_pdf, format="pdf", dpi=300)
    plt.close(fig)

    return {"cellline": cellline, "pearson_r": r, "r2": r2, "n_genes": int(x.size), "out_pdf": str(out_pdf)}

# for Figure 3: delta log-expression heatmaps comparing Teton vs Bulk
def plot_logfc_heatmap_2col(
    expr_bulk_log,   # genes x conditions
    expr_teton_log,  # genes x conditions
    a, b,
    genes=None,
    n_top=8,
    min_abs_logfc=0.2,
    rank_mode="min_abs",
    order_by="bulk",
    title=None,
    clip=2.0,
    savepath=None,
    dpi=300
):
    shared = expr_bulk_log.index.intersection(expr_teton_log.index)
    bulk = expr_bulk_log.loc[shared, [a, b]].astype(float)
    tet  = expr_teton_log.loc[shared, [a, b]].astype(float)

    lfc_bulk  = bulk[a] - bulk[b]
    lfc_teton = tet[a]  - tet[b]

    if genes is not None:
        genes_use = [g for g in genes if g in lfc_bulk.index and g in lfc_teton.index]
        if len(genes_use) == 0:
            raise ValueError("None of the requested genes are present in both datasets.")
        mat = pd.DataFrame({"Bulk": lfc_bulk.loc[genes_use], "Teton": lfc_teton.loc[genes_use]})
    else:
        agree = np.sign(lfc_bulk) == np.sign(lfc_teton)
        strength = np.minimum(lfc_bulk.abs(), lfc_teton.abs()) if rank_mode == "min_abs" else (lfc_bulk.abs() + lfc_teton.abs()) / 2
        keep = agree & (lfc_bulk.abs() >= min_abs_logfc) & (lfc_teton.abs() >= min_abs_logfc)
        top_genes = strength[keep].sort_values(ascending=False).head(n_top).index
        mat = pd.DataFrame({"Bulk": lfc_bulk.loc[top_genes], "Teton": lfc_teton.loc[top_genes]})

    order_score = mat["Bulk"] if order_by == "bulk" else mat.mean(axis=1)
    mat = mat.loc[order_score.sort_values(ascending=False).index]

    vmax = float(np.nanmax(np.abs(mat.to_numpy())))
    if clip is not None:
        vmax = min(vmax, clip)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

    fig, ax = plt.subplots(figsize=FIG_SIZE)
    im = ax.imshow(mat.to_numpy(), aspect="auto", interpolation="nearest", norm=norm, cmap="RdBu_r")

    ax.set_yticks(np.arange(len(mat.index)))
    ax.set_yticklabels(mat.index)
    ax.set_xticks(np.arange(mat.shape[1]))
    ax.set_xticklabels(mat.columns)
    ax.set_title(title or f"{a} vs {b}", pad=2)

    # gridlines
    ax.set_xticks(np.arange(-0.5, mat.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, mat.shape[0], 1), minor=True)
    ax.grid(which="minor", color="black", linewidth=0.6)
    ax.tick_params(which="minor", bottom=False, left=False)

    cbar = fig.colorbar(im, ax=ax, fraction=0.06, pad=0.04)
    cbar.set_label("Δ log expression", fontsize=6)
    cbar.ax.tick_params(labelsize=6)

    fig.tight_layout(pad=0.5)

    if savepath:
        savepath = Path(savepath)
        fig.savefig(savepath, format="pdf", dpi=dpi, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved: {savepath}")
    else:
        plt.show()

    return mat

# builds expression matrices for Fig3 heatmaps
def build_bulk_log(bulk_tpm_df, bulk_col_map):
    cols = {cl: bulk_col_map.get(cl, cl) for cl in ["HeLa", "A549", "HUVEC"]}
    for cl, c in cols.items():
        if c not in bulk_tpm_df.columns:
            raise KeyError(f"Bulk column missing for {cl}: expected '{c}'")
    bulk_counts = bulk_tpm_df[[cols["HeLa"], cols["A549"], cols["HUVEC"]]].copy()
    bulk_counts.columns = ["HeLa", "A549", "HUVEC"]  # genes x celllines
    return np.log10(bulk_counts.astype(float) + 1.0)


def build_teton_log(results_dict, agg="mean"):
    rows = {}
    for cl in ["HeLa", "A549", "HUVEC"]:
        v = teton_pseudobulk_from_wells(results_dict, cl, wells=None, agg=agg)
        rows[cl] = np.log10(v.astype(float) + 1.0)
    return pd.DataFrame(rows).T


##### Run the full analysis and generate ouputs and figures
This is the main pipeline: <br>
1. Create output folders
2. Process each Teton run into into pseudo-bulk matrices
3. Generate Fig1: Wells A1 vs F2 scatter per run
4. Load bulk TPM and generate Fig2: Bulk vs Teton scatter per cell line
5. Generate Fig3: heatmaps of delta log-expression comparing Teton vs Bulk for selected genes

In [ ]:
def main():
    output_dir.mkdir(parents=True, exist_ok=True)
    fig_dir.mkdir(parents=True, exist_ok=True)

    results = {}
    for name in RUNS:
        print("\n====================")
        print("Processing:", name)

        results[name] = process_run(
            RUNS[name],
            PANELS[name],
            group_col=GROUP_COL,
            report_batches=False
        )

        results[name]["well_cp10k_gene"].to_parquet(output_dir / f"{name}_well_cp10k_gene.parquet")
        results[name]["well_log10_gene"].to_parquet(output_dir / f"{name}_well_log10_gene.parquet")

    print("Done processing all runs.")

    ### Fig1: A1 vs F2 scatter
    for name in RUNS.keys():
        well_log10_gene = results[name]["well_log10_gene"]
        lims = (np.nanmin(well_log10_gene.to_numpy()), np.nanmax(well_log10_gene.to_numpy()))
        out_pdf = fig_dir / f"Fig1_{name}_A1_vs_F2_scatter.pdf"
        plot_well_scatter_pdf(well_log10_gene, "A1", "F2", out_pdf, lims=lims)
        print("Saved:", out_pdf)

    bulk_tpm = load_bulk_tpm(bulk_path)
    print("bulk_tpm:", bulk_tpm.shape)

    ### Fig2: Bulk vs Teton
    aligned_cache = {}
    all_xy = []

    for cellline in RUNS.keys():
        bulk_col = BULK_COL_MAP[cellline]
        teton_vec = teton_pseudobulk_from_wells(results, cellline, wells=None, agg="mean")
        tlog, blog = align_teton_bulk(teton_vec, bulk_tpm, bulk_col)

        aligned_cache[cellline] = (tlog, blog)
        all_xy.append(tlog.to_numpy())
        all_xy.append(blog.to_numpy())

    all_xy = np.concatenate(all_xy)
    lims = (np.nanmin(all_xy), np.nanmax(all_xy))

    fig2_rows = []
    for cellline, (tlog, blog) in aligned_cache.items():
        out_pdf = fig_dir / f"Fig2_{cellline}_Teton_vs_Bulk.pdf"
        row = plot_teton_vs_bulk_pdf(tlog, blog, cellline, out_pdf, lims=lims)
        fig2_rows.append(row)
        print("Saved:", out_pdf)

    pd.DataFrame(fig2_rows).to_csv(fig_dir / "Fig2_Teton_vs_Bulk_summary.csv", index=False)

    ### Fig3 heatmaps
    bulk_log = build_bulk_log(bulk_tpm, BULK_COL_MAP)
    teton_log = build_teton_log(results)

    shared = bulk_log.index.intersection(teton_log.columns)
    expr_bulk_log = bulk_log.loc[shared]
    expr_teton_log = teton_log[shared].T

    plot_logfc_heatmap_2col(
        expr_bulk_log, expr_teton_log,
        a="HUVEC", b="A549",
        genes=["KDR", "FLT1", "TEK", "MEF2C", "MET"],
        title="HUVEC vs A549",
        savepath=fig_dir / "Fig3_HUVEC_vs_A549_grid_specificGenes.pdf"
    )

    plot_logfc_heatmap_2col(
        expr_bulk_log, expr_teton_log,
        a="HUVEC", b="HeLa",
        genes=["CDKN2A", "MET", "KDR", "FLT1", "TEK"],
        title="HUVEC vs HeLa",
        savepath=fig_dir / "Fig3_HUVEC_vs_HeLa_grid_specificGenes.pdf"
    )

    plot_logfc_heatmap_2col(
        expr_bulk_log, expr_teton_log,
        a="HeLa", b="A549",
        genes=["CDKN2A", "SKP2", "IL1R1", "TNFRSF1A", "DUSP5"],
        title="HeLa vs A549",
        savepath=fig_dir / "Fig3_HeLa_vs_A549_grid_specificGenes.pdf"
    )

    print("\nAll figures written to:", fig_dir)


if __name__ == "__main__":
    main()